# 红利低波数据下载

本 notebook 只负责数据更新。参数来自 `config.yaml`；每次只启动一个下载任务。

In [1]:
import os
import sys
from pathlib import Path

project_root = next(
    path
    for path in (Path.cwd(), *Path.cwd().parents)
    if (path / "pyproject.toml").exists()
)
os.chdir(project_root)
source_root = project_root / "src"
if str(source_root) not in sys.path:
    sys.path.insert(0, str(source_root))

In [2]:
from importlib.resources import files

import pandas as pd
import yaml

from pyquant import (
    get_period_end_dates,
    load_dataset,
    update_dataset,
    update_minute_data,
)
from strategies.dividend_low_vol.components import (
    build_intraday_minute_requests,
    select_dividend_low_vol_candidates,
)
from strategies.dividend_low_vol.components import (
    select_dividend_low_vol_download_symbols,
)

with (
    files("strategies.dividend_low_vol")
    .joinpath("config.yaml")
    .open(encoding="utf-8") as stream
):
    config = yaml.safe_load(stream) or {}
start_date = pd.Timestamp(config["data"]["start_date"])
end_date = pd.Timestamp(config["data"]["end_date"])
pool = config["data"]["pool"]
lookback_date = start_date - pd.DateOffset(years=3)
dividend_start = (
    lookback_date - pd.DateOffset(years=1)
    if (start_date.month == 12 and start_date.day <= 20)
    else lookback_date
)

## 官方中证指数

此下载独立于 BaoStock，不使用其请求限额、下载锁或下载控制单元。

In [ ]:
official_index_job = update_dataset(
    "csindex_daily",
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=["H30269", "H20269"],
)
official_index_job.wait()

## 官方历史成份股

该单元在当前 notebook 内核调用 RQData。请由运行者自行选择已配置 RQData 认证的环境。目标文件已存在时默认拒绝覆盖。

In [ ]:
constituent_job = update_dataset(
    "index_constituents",
    start=start_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=[config["strategy_3"]["index_code"]],
)
constituent_snapshots = constituent_job.wait()

## RQData 六口径 PB

通过 RQData `get_factor` 下载六种 PB 口径。股票池由 RQData 按下载区间解析为历史全市场普通股，包含区间内退市股票；请求按缺失区间增量补齐。请在已配置 RQData 认证的环境中运行。

In [3]:
valuation_start = start_date.to_period("M").start_time
pb_job = update_dataset(
    "stock_pb_daily",
    start=valuation_start.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool="all",
)
pb_downloads = pb_job.wait()

Updated 0/0

RuntimeError: RQData instrument-list request failed: Quota exceeded

## 日行情

先运行此单元格并等待完成，再生成分红和股本下载股票池。

In [ ]:
download_job = update_dataset(
    "stock_daily",
    start=lookback_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=pool,
)

### 下载控制

In [ ]:
download_job.pause()
download_job.state

In [ ]:
download_job.resume()
download_job.state

In [ ]:
download_job.stop()
download_job.wait()
download_job.state

## 分红与股本下载股票池

按完整下载区间内至少 720 条有效行情筛选；结果覆盖 `pool`，供分红和股本共用。

In [4]:
price = load_dataset(
    "stock_daily",
    start=lookback_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
)
pool = select_dividend_low_vol_download_symbols(price, end_date, config)
if not pool:
    raise ValueError("No symbols have at least 720 valid prices in the download range")
print(f"Dividend/share download pool: {len(pool)} symbols")

Dividend/share download pool: 3922 symbols


## 分红数据

In [ ]:
download_job = update_dataset(
    "dividend",
    start=dividend_start.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=pool,
)

## 季度总股本

In [ ]:
download_job = update_dataset(
    "stock_profit_quarterly",
    start=lookback_date.strftime("%Y-%m-%d"),
    end=end_date.strftime("%Y-%m-%d"),
    pool=pool,
)

## 1 分钟行情

确认日行情、分红和季度总股本任务均已完成后再运行。本区段按每个调仓月末的股息率前 150 名候选，下载其过去 20 个交易日的未复权 1 分钟行情。

In [ ]:
dividends = load_dataset("dividend")
dividend_queries = load_dataset("dividend_queries")
shares = load_dataset("stock_profit_quarterly")
trading_dates = price["date"].drop_duplicates().sort_values()
rebalance_dates = get_period_end_dates(
    trading_dates[trading_dates.between(start_date, end_date)]
)
candidate_config = {"universe": config["universe"], "selection": config["strategy_1"]}
minute_config = config["minute_data"]
minute_requests = []
for signal_date in rebalance_dates:
    candidates = select_dividend_low_vol_candidates(
        price, dividends, dividend_queries, shares, signal_date, candidate_config
    )
    minute_requests.extend(
        build_intraday_minute_requests(
            candidates.index.tolist(),
            signal_date,
            trading_dates,
            lookback_trading_days=minute_config["lookback_trading_days"],
            max_candidates=minute_config["max_candidates"],
        )
    )
if not minute_requests:
    raise ValueError("No minute-data requests were generated")
print(f"Minute-data requests: {len(minute_requests)}")

In [ ]:
minute_job = update_minute_data(
    minute_requests,
    max_attempts=minute_config["max_attempts"],
    quota_reserve_bytes=minute_config["quota_reserve_bytes"],
    min_bars_per_day=minute_config["min_bars_per_day"],
)

In [ ]:
minute_downloads = minute_job.wait()
minute_downloads["status"].value_counts(dropna=False)